<a href="https://colab.research.google.com/github/Minhaz-ck/First/blob/main/EV_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
upload=files.upload()

In [ ]:
import pandas as pd
import numpy as np
ev=pd.read_csv('train.csv')
ev.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


In [ ]:
ev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 668665 entries, 0 to 668664
Data columns (total 15 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           668665 non-null  int64  
 1   Age                          668665 non-null  int64  
 2   Annual_Income_USD            668665 non-null  float64
 3   Daily_Commute_km             668665 non-null  float64
 4   Number_of_Cars_Owned         668665 non-null  int64  
 5   Charging_Stations_Near_Home  668665 non-null  int64  
 6   Charging_Stations_Near_Work  668665 non-null  int64  
 7   Environmental_Concern_Level  668665 non-null  float64
 8   Gender                       668665 non-null  object 
 9   City_Type                    668665 non-null  object 
 10  Current_Car_Type             668665 non-null  object 
 11  Home_Charging_Possible       668665 non-null  object 
 12  Subsidy_Available            668665 non-null  object 
 13 

In [ ]:
ev_labels=ev["Will_Buy_EV"].copy()
ev=ev.drop('Will_Buy_EV', axis=1)

In [ ]:
ev_num=ev.select_dtypes(include=[np.number])

In [ ]:
from sklearn.ensemble import IsolationForest
model=IsolationForest(random_state=42)
out_ped=model.fit_predict(ev_num)

In [ ]:
ev=ev.iloc[out_ped==1]
ev_labels=ev_labels.iloc[out_ped==1]

In [ ]:
ev['Current_Car_Type'].value_counts()

,count
Current_Car_Type,
Sedan,303459
SUV,246545
Hatchback,79438
Truck,39223


In [ ]:
from sklearn.preprocessing import OneHotEncoder
cat_end = OneHotEncoder(sparse_output=False)
ev_cat = ev[["Gender","City_Type","Current_Car_Type","Home_Charging_Possible","Subsidy_Available","Range_Anxiety_Level"]]
ev_cat_1hot = cat_end.fit_transform(ev_cat)

In [ ]:
ev_cat_1hot

array([[0., 1., 0., ..., 0., 1., 0.],
       [0., 1., 0., ..., 0., 1., 0.],
       [0., 1., 0., ..., 0., 1., 0.],
       ...,
       [1., 0., 0., ..., 0., 1., 0.],
       [1., 0., 0., ..., 0., 1., 0.],
       [0., 1., 0., ..., 0., 1., 0.]])

In [ ]:
cat_end.feature_names_in_

array(['Gender', 'City_Type', 'Current_Car_Type',
       'Home_Charging_Possible', 'Subsidy_Available',
       'Range_Anxiety_Level'], dtype=object)

In [ ]:
cat_end.get_feature_names_out()

array(['Gender_Female', 'Gender_Male', 'Gender_Other', 'City_Type_Rural',
       'City_Type_Suburban', 'City_Type_Urban',
       'Current_Car_Type_Hatchback', 'Current_Car_Type_SUV',
       'Current_Car_Type_Sedan', 'Current_Car_Type_Truck',
       'Home_Charging_Possible_No', 'Home_Charging_Possible_Yes',
       'Subsidy_Available_No', 'Subsidy_Available_Yes',
       'Range_Anxiety_Level_High', 'Range_Anxiety_Level_Low',
       'Range_Anxiety_Level_Medium'], dtype=object)

In [ ]:
from sklearn.preprocessing import StandardScaler
std_scaler=StandardScaler()
ev_num_std_scaler = std_scaler.fit_transform(ev_num)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
num_pipeline = make_pipeline(SimpleImputer(strategy = "median"),StandardScaler())
cat_pipeline = make_pipeline(SimpleImputer(strategy = "most_frequent"), OneHotEncoder(handle_unknown="ignore"))

In [ ]:
from sklearn.compose import make_column_selector, make_column_transformer
preprocessing = make_column_transformer(
                   (num_pipeline,make_column_selector(dtype_include=np.number)),
                   (cat_pipeline,make_column_selector(dtype_include=object)),
)

In [ ]:
ev_p=preprocessing.fit_transform(ev)
ev_p.shape

(305773, 25)

In [ ]:
preprocessing.get_feature_names_out()

array(['pipeline-1__id', 'pipeline-1__Age',
       'pipeline-1__Annual_Income_USD', 'pipeline-1__Daily_Commute_km',
       'pipeline-1__Number_of_Cars_Owned',
       'pipeline-1__Charging_Stations_Near_Home',
       'pipeline-1__Charging_Stations_Near_Work',
       'pipeline-1__Environmental_Concern_Level',
       'pipeline-2__Gender_Female', 'pipeline-2__Gender_Male',
       'pipeline-2__Gender_Other', 'pipeline-2__City_Type_Rural',
       'pipeline-2__City_Type_Suburban', 'pipeline-2__City_Type_Urban',
       'pipeline-2__Current_Car_Type_Hatchback',
       'pipeline-2__Current_Car_Type_SUV',
       'pipeline-2__Current_Car_Type_Sedan',
       'pipeline-2__Current_Car_Type_Truck',
       'pipeline-2__Home_Charging_Possible_No',
       'pipeline-2__Home_Charging_Possible_Yes',
       'pipeline-2__Subsidy_Available_No',
       'pipeline-2__Subsidy_Available_Yes',
       'pipeline-2__Range_Anxiety_Level_High',
       'pipeline-2__Range_Anxiety_Level_Low',
       'pipeline-2__Range_Anxie

LinReg Model

The `Will_Buy_EV` column (stored in `ev_labels`) contains categorical values ('No', 'Yes'). For a `LinearRegression` model, the target variable must be numerical. We need to convert these labels to a numerical representation, typically 0 and 1.

In [ ]:
ev_labels_num = ev_labels.map({'No': 0, 'Yes': 1})
display(ev_labels_num.value_counts())

,count
Will_Buy_EV,
0,258425
1,47348


In [ ]:
from sklearn.linear_model import LinearRegression
l_model = make_pipeline(preprocessing, LinearRegression())
l_model.fit(ev,ev_labels_num)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipeline-1',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x79e333b8f8c0>),
                                                 ('pipeline-2',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x79e333c1a990>)])),
                ('linearregression', LinearRegression())])

In [ ]:
ev_ped = l_model.predict(ev)
ev_ped[:5]

array([ 0.01499872, -0.02422895,  0.01255674,  0.30229389,  0.05849373])

In [ ]:
from google.colab import files
upload=files.upload()

Saving test.csv to test.csv


In [ ]:
ev_test = pd.read_csv("test.csv")

In [ ]:
ev_ped_l = l_model.predict(ev_test)

In [ ]:
test_ids = ev_test["id"]

In [ ]:
submission7 = pd.DataFrame({
    'id': test_ids,          # your saved test id column
    'Will_Buy_EV': ev_ped_l   # replace 'target' with actual competition target column name
})

submission7.to_csv('submission7.csv', index=False)

In [ ]:
submission6.head()

,id,Will_Buy_EV
0,668665,0.154847
1,668666,0.154847
2,668667,0.154847
3,668668,0.154847
4,668669,0.154847


In [ ]:
ev_labels_num.iloc[:5].values

array([0, 0, 0, 1, 0])

In [ ]:
from sklearn.tree import DecisionTreeRegressor
t_model = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
t_model.fit(ev,ev_labels_num)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipeline-1',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x785c6e4d7cb0>),
                                                 ('pipeline-2',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x785c6e462c10>)])),
                ('decisiontreeregressor',
                 DecisionTreeRegressor(random_state=42))])

In [ ]:
ev_ped = t_model.predict(ev)
ev_ped[3145:3150]

array([1., 0., 0., 0., 0.])

In [ ]:
ev_labels_num.iloc[3145:3150].values

array([1, 0, 0, 0, 0])

In [ ]:
from sklearn.metrics import root_mean_squared_error
t_rmse=root_mean_squared_error(ev_labels_num,ev_ped)
t_rmse

0.0

In [ ]:
from sklearn.model_selection import cross_val_score
t_rmses = -cross_val_score(t_model,ev,ev_labels_num,scoring = "neg_root_mean_squared_error",cv=10)

In [ ]:
pd.Series(t_rmses).describe()

,0
count,10.000000
mean,0.395927
std,0.022847
min,0.381104
25%,0.383856
50%,0.385454
75%,0.390259
max,0.448477


In [ ]:
from sklearn.ensemble import RandomForestRegressor
f_reg=make_pipeline(preprocessing,RandomForestRegressor(random_state=42))
f_remses = -cross_val_score(f_reg,ev,ev_labels_num,scoring="neg_root_mean_squared_error",cv=5,n_jobs=-1)

In [ ]:
f_remses

array([0.28812584, 0.27712611, 0.27299422, 0.27503903, 0.30608939])

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from scipy.stats import randint
full_pipeline = Pipeline([("preprocessing",preprocessing),
                          ("random_forest",RandomForestRegressor(random_state=42))
                         ])
param_distribs = {"random_forest__n_estimators":randint(low=3,high=50),
                  "random_forest__max_features":randint(low=2,high=20)}
rnd_search = RandomizedSearchCV(full_pipeline, param_distributions = param_distribs, n_iter=10,cv=3,
                                scoring="neg_root_mean_squared_error", random_state=42)
rnd_search.fit(ev,ev_labels_num)

RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(transformers=[('pipeline-1',
                                                                               Pipeline(steps=[('simpleimputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('standardscaler',
                                                                                                StandardScaler())]),
                                                                               <sklearn.compose._column_transformer.make_column_selector object at 0x785c6e4d7cb0>),
                                                                              ('pipeline-2',
                                                                               Pipeline(steps=[('simpleimputer',
                                                                                                Sim...
                                                                               <sklearn.compose._column_transformer.make_column_selector object at 0x785c6e462c10>)])),
                                             ('random_forest',
                                              RandomForestRegressor(random_state=42))]),
                   param_distributions={'random_forest__max_features': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x785c6e1e4b50>,
                                        'random_forest__n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x785c6e1e4650>},
                   random_state=42, scoring='neg_root_mean_squared_error')

In [ ]:
final_model = rnd_search.best_estimator_

In [ ]:
ev_test.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level
0,668665,61,67725.0,16.9,2,7,4,4.0,Male,Suburban,Sedan,Yes,No,Low
1,668666,42,152835.0,41.9,2,9,9,4.0,Male,Urban,SUV,No,No,Low
2,668667,68,86877.0,53.3,1,10,11,4.0,Female,Urban,Sedan,No,No,Low
3,668668,39,46794.0,34.1,2,4,8,4.0,Female,Suburban,Sedan,Yes,No,Low
4,668669,55,112172.0,57.2,1,6,4,1.0,Female,Suburban,Sedan,Yes,Yes,Low


In [ ]:
final_ped = final_model.predict(ev_test)

In [ ]:
final_ped

array([0.        , 0.16666667, 0.02380952, ..., 0.0952381 , 0.02380952,
       0.        ])

In [ ]:

submission = pd.DataFrame({
    'id': test_ids,          # your saved test id column
    'Will_Buy_EV': final_ped   # replace 'target' with actual competition target column name
})

submission.to_csv('submission.csv', index=False)

In [ ]:
submission.head()

,id,Will_Buy_EV
0,668665,0.000000
1,668666,0.166667
2,668667,0.023810
3,668668,0.071429
4,668669,0.119048


In [ ]:
sec_ped = l_model.predict(ev_test)
sec_ped

array([ 0.11483158,  0.28637997,  0.1256224 , ...,  0.06336661,
       -0.13916217, -0.21839884])

In [ ]:
third_ped = t_model.predict(ev_test)
third_ped

array([0., 0., 0., ..., 0., 0., 0.])

In [ ]:
f_reg.fit(ev,ev_labels_num)
f_reg.predict(ev_test)

array([0.04, 0.05, 0.  , ..., 0.14, 0.  , 0.  ])

In [ ]:
from sklearn.linear_model import SGDClassifier
sgd_clf = make_pipeline(preprocessing, SGDClassifier(random_state=42))
sgd_clf.fit(ev,ev_labels_num)
sgd_clf.predict(ev_test)

array([0, 0, 0, ..., 0, 0, 0])

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from scipy.stats import randint, uniform
full_pipeline = Pipeline([("preprocessing",preprocessing),
                          ("gb_reg",GradientBoostingRegressor(random_state=42))
                         ])
param_distribs = {
    "gb_reg__n_estimators": randint(low=50, high=200),     # More trees usually help GB
    "gb_reg__learning_rate": uniform(loc=0.01, scale=0.2), # Controls how fast it learns
    "gb_reg__max_depth": randint(low=3, high=8)            # Trees are usually shallower than RF
}
rnd_search = RandomizedSearchCV(full_pipeline, param_distributions = param_distribs, n_iter=10,cv=3,
                                scoring="neg_root_mean_squared_error", random_state=42)
rnd_search.fit(ev,ev_labels_num)

RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(transformers=[('pipeline-1',
                                                                               Pipeline(steps=[('simpleimputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('standardscaler',
                                                                                                StandardScaler())]),
                                                                               <sklearn.compose._column_transformer.make_column_selector object at 0x785c6e4d7cb0>),
                                                                              ('pipeline-2',
                                                                               Pipeline(steps=[('simpleimputer',
                                                                                                Sim...
                   param_distributions={'gb_reg__learning_rate': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x785c6e1d5450>,
                                        'gb_reg__max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x785c6e19b410>,
                                        'gb_reg__n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x785c7eca87d0>},
                   random_state=42, scoring='neg_root_mean_squared_error')

In [ ]:
best_model = rnd_search.best_estimator_

In [ ]:
gbr_ped =best_model.predict(ev_test)
gbr_ped

array([ 0.02631414,  0.12608387, -0.00028071, ...,  0.01494611,
        0.02243769, -0.00655738])

In [ ]:
submission2 = pd.DataFrame({
    'id': test_ids,          # your saved test id column
    'Will_Buy_EV': gbr_ped   # replace 'target' with actual competition target column name
})

submission2.to_csv('submission1.csv', index=False)

In [ ]:
submission2.head()

,id,Will_Buy_EV
0,668665,0.028525
1,668666,0.127527
2,668667,-0.014770
3,668668,0.051365
4,668669,0.083681


In [ ]:
!pip install xgboost

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from scipy.stats import randint, uniform
full_pipeline = Pipeline([("preprocessing",preprocessing),
                          ("xgb_model",XGBRegressor(random_state=42))
                         ])
param_distribs = {
    "xgb_model__n_estimators": randint(low=50, high=200),     # More trees usually help GB
    "xgb_model__learning_rate": uniform(loc=0.01, scale=0.2), # Controls how fast it learns
    "xgb_model__max_depth": randint(low=3, high=10),            # Trees are usually shallower than RF
    "xgb_model__reg_lambda": uniform(loc=0.1, scale=2.0),
}
xgb_search = RandomizedSearchCV(full_pipeline, param_distributions = param_distribs, n_iter=10,cv=3,
                                scoring="neg_root_mean_squared_error", random_state=42)
xgb_search.fit(ev,ev_labels_num)

RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(transformers=[('pipeline-1',
                                                                               Pipeline(steps=[('simpleimputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('standardscaler',
                                                                                                StandardScaler())]),
                                                                               <sklearn.compose._column_transformer.make_column_selector object at 0x785c6e4d7cb0>),
                                                                              ('pipeline-2',
                                                                               Pipeline(steps=[('simpleimputer',
                                                                                                Sim...
                                        'xgb_model__max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x785c602ea530>,
                                        'xgb_model__n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x785c602e9270>,
                                        'xgb_model__reg_lambda': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x785c602ef650>},
                   random_state=42, scoring='neg_root_mean_squared_error')

In [ ]:
best_model2 = xgb_search.best_estimator_

In [ ]:
xgb_ped =best_model2.predict(ev_test)
xgb_ped

array([ 0.02852477,  0.12752719, -0.01477043, ...,  0.05406706,
       -0.00696563, -0.01315459], dtype=float32)

In [ ]:
submission3.head()

,id,Will_Buy_EV
0,668665,0.028525
1,668666,0.127527
2,668667,-0.014770
3,668668,0.051365
4,668669,0.083681


In [ ]:
submission3 = pd.DataFrame({
    'id': test_ids,          # your saved test id column
    'Will_Buy_EV': xgb_ped   # replace 'target' with actual competition target column name
})

submission3.to_csv('submission3.csv', index=False)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from scipy.stats import randint, uniform
full_pipeline = Pipeline([("preprocessing",preprocessing),

                          ("hgb_reg",HistGradientBoostingRegressor(random_state=42,
                            early_stopping=False,          # Stops training early if validation score stops improving
        categorical_features="from_dtype" # Leverages native categorical handling if applicable
               ))
                         ])

param_distribs = {
    "hgb_reg__max_iter": randint(low=200, high=1500),         # Increased upper limit for better learning
    "hgb_reg__learning_rate": uniform(loc=0.005, scale=0.197), # Slightly tighter learning rate
    "hgb_reg__max_leaf_nodes": randint(low=31, high=255),    # Allows deeper trees if data supports it
    "hgb_reg__min_samples_leaf": randint(low=20, high=200),  # Prevents overfitting in individual leaves
    "hgb_reg__l2_regularization": uniform(loc=0.0, scale=20.0), # L2 regularization (very effective)
    "hgb_reg__max_bins": randint(low=128, high=255)               # Limits maximum tree depth explicitly
}

hgd_search = RandomizedSearchCV(full_pipeline, param_distributions = param_distribs, n_iter=50,cv=5,
                                scoring="neg_root_mean_squared_error", random_state=42)
hgd_search.fit(ev,ev_labels_num)

In [ ]:
best_model3 = hgd_search.best_estimator_

In [ ]:
hgb_ped =best_model3.predict(ev_test)
hgb_ped

array([0.01263564, 0.03480477, 0.0059397 , ..., 0.00765452, 0.00134826,
       0.00382588])

In [ ]:
submission4 = pd.DataFrame({
    'id': test_ids,          # your saved test id column
    'Will_Buy_EV': hgb_ped   # replace 'target' with actual competition target column name
})

submission4.to_csv('submission8.csv', index=False)

In [ ]:
submission4.head()

,id,Will_Buy_EV
0,668665,0.012636
1,668666,0.034805
2,668667,0.005940
3,668668,0.003317
4,668669,0.016533
